## Intelligent document processor Chat Bot

Although this was created in Google Colab, I ensured the program would run by executing only one cell for demo purposes. Please keep in mind that you need to enter your own GEMINI API key to receive responses from the LLM. These can be generated for free with limited use via Google AI Studio. Instructions on how to enter your api key will be listed below this cell.

Google Ai Studio link here: https://aistudio.google.com/welcome

In [ ]:
# Install dependencies
!pip install -q pymupdf
!pip install -q llama-index
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-llms-google-genai
!pip -q install nest_asyncio

In [ ]:
# Import libraries
import os
from google.colab import files, userdata

import fitz # pymupdf

from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import VectorIndexRetriever

from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.llms.google_genai import GoogleGenAI

import nest_asyncio

In [ ]:
import gradio as gr

# FILE -> INDEX -> RETRIEVER(k) -> QUERY ENGINE -> CHAT
text_state = gr.State()
index_state = gr.State()
query_engine_state = gr.State()

# Turns PDF files into a string of text
def load_file(file):
  doc = fitz.open(file.name)
  text = "\n".join([page.get_text() for page in doc])
  return text

  print(f"Uploaded PDF: {pdf_path}")
  print(f"Extracted {len(text.split())} words")

def build_index(text, chunk_size, chunk_overlap):
  # Chunking
  chunk_size = chunk_size
  chunk_overlap = chunk_overlap

  splitter = SentenceSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)

  # Document Obj
  documents = [Document(text=text)]
  nodes = splitter.get_nodes_from_documents(documents)
  print("Total chunks (nodes): ", len(nodes))

  index = VectorStoreIndex(nodes)
  return index

def build_query_engine(index, k_depth):

  k = k_depth # retrieval depth

  retriever = VectorIndexRetriever(index=index, similarity_top_k=k)
  print("Retriever ready:", {"method": "vector", "top_k": k})

  synthesizer = get_response_synthesizer(
    response_mode="compact",
  )

  query_engine = RetrieverQueryEngine(
      retriever=retriever,
      response_synthesizer=synthesizer,
  )
  return query_engine

# Chat history
history = []


def chat(message, chat_history, query_engine):
  response = query_engine.query(message) # chat_interface handles history 
  return response


with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
  with gr.Row():
    with gr.Column(scale=1):
      labelA = gr.Label("Upload Media & Parameter Tuning", show_label=False)
      file_upload = gr.File(show_label=False)
      file_upload.change(
        fn=load_file,
        inputs=[file_upload],
        outputs=text_state,
    )

      chunk_size = gr.Number(label="Chunk size")
      chunk_overlap = gr.Number(label="Overlap")
      chunk_size.change(
        fn=build_index,
        inputs=[text_state, chunk_size, chunk_overlap],
        outputs=index_state
    )

      k_slider = gr.Slider(minimum=1, maximum=10, step=1,
                        label="Adjust depth (k) value")
      k_slider.change(
        fn=build_query_engine,
        inputs=[index_state, k_slider],
        outputs=query_engine_state,
    )

    with gr.Column(scale=2):
      chat_interface = gr.ChatInterface(
        fn=chat,
        type="messages",
        chatbot=gr.Chatbot(
            height=604,
            show_label=False,
            type="messages",
            allow_tags=True,
        ),
        additional_inputs=[query_engine_state],
        textbox=gr.Textbox(
            placeholder="Type here..."
        ),
        autoscroll=True
    )

    demo.launch()